# H2SNN Paper Studies

Colab runner for H2SNN-only ablation configs, result collection, plots, and diagnostics. This notebook does not create baseline models.

## 1. Mount Google Drive And Enter Repo

Edit `PROJECT_DIR` if your repo lives somewhere else.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/H2S-NN_test'
%cd $PROJECT_DIR
!pwd


## 2. Install Dependencies

Colab already includes PyTorch. This cell installs PyG wheels matching the active Torch/CUDA build.

In [ ]:
import torch

torch_version = torch.__version__.split('+')[0]
cuda_tag = 'cpu' if torch.version.cuda is None else 'cu' + torch.version.cuda.replace('.', '')
wheel_url = f'https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html'
print('torch:', torch.__version__)
print('pyg wheel url:', wheel_url)

!pip -q install pandas matplotlib pyyaml scikit-learn
!pip -q install torch-geometric torch-scatter torch-sparse -f $wheel_url


## 3. CLI Sanity Check

In [ ]:
import torch, torch_geometric
print('cuda available:', torch.cuda.is_available())
print('torch_geometric:', torch_geometric.__version__)
!python scripts/h2snn_paper_studies.py --help


## 4. Generate A Small Smoke Manifest

This only writes YAML configs and a manifest. It does not train. Increase datasets, seeds, and epochs for paper-grade runs.

In [ ]:
MANIFEST = 'configs/generated/h2snn_paper_ablation_colab/manifest.csv'
EXPERIMENTS = 'experiments/h2snn_paper_ablation_colab'

!python scripts/h2snn_paper_studies.py make-configs \
  --datasets acm \
  --mode manual \
  --suite core \
  --seeds 42 \
  --epochs 5 \
  --device cuda \
  --out-dir configs/generated/h2snn_paper_ablation_colab \
  --output-dir $EXPERIMENTS \
  --manifest $MANIFEST

import pandas as pd
pd.read_csv(MANIFEST)


## 5. Optional Tiny Training Smoke Run

Runs at most one generated config. Delete `--max-runs 1` when you are ready to run the full manifest.

In [ ]:
# !python scripts/h2snn_paper_studies.py run \
#   --manifest $MANIFEST \
#   --device cuda \
#   --max-runs 1 \
#   --skip-existing


## 6. Collect Metrics And Make Plots

Run after some experiments have completed.

In [ ]:
SUMMARY_CSV = f'{EXPERIMENTS}/summary.csv'
FIG_DIR = 'figures/h2snn_paper_ablation_colab'

# !python scripts/h2snn_paper_studies.py collect \
#   --experiments-dir $EXPERIMENTS \
#   --manifest $MANIFEST \
#   --out-csv $SUMMARY_CSV
# !python scripts/h2snn_paper_studies.py plot \
#   --summary-csv $SUMMARY_CSV \
#   --out-dir $FIG_DIR \
#   --metric best_test_macro_f1


## 7. Dataset And Face Diagnostics

These commands load a dataset and build diagnostic CSV/PNG files. Use manual and auto configs separately.

In [ ]:
# !python scripts/h2snn_paper_studies.py diagnostics \
#   --config configs/experiments/acm_h2snn.yaml \
#   --out-dir figures/h2snn_paper_diagnostics_colab/acm_manual
# !python scripts/h2snn_paper_studies.py diagnostics \
#   --config configs/experiments/acm_h2snn_ampss.yaml \
#   --out-dir figures/h2snn_paper_diagnostics_colab/acm_auto


## 8. Paper-Grade Template

Use this once the smoke path works. It writes configs for ACM/DBLP/IMDB, five seeds, and the core+sweep suites.

In [ ]:
# PAPER_MANIFEST = 'configs/generated/h2snn_paper_ablation_full/manifest.csv'
# PAPER_EXPERIMENTS = 'experiments/h2snn_paper_ablation_full'
# !python scripts/h2snn_paper_studies.py make-configs \
#   --datasets acm dblp imdb \
#   --mode manual \
#   --suite all \
#   --seeds 42 43 44 45 46 \
#   --epochs 200 \
#   --device cuda \
#   --out-dir configs/generated/h2snn_paper_ablation_full \
#   --output-dir $PAPER_EXPERIMENTS \
#   --manifest $PAPER_MANIFEST
# !python scripts/h2snn_paper_studies.py run --manifest $PAPER_MANIFEST --device cuda --skip-existing
